# Writing to Apache Iceberg

Iceberg is another transactional table format. Unlike Delta, Iceberg is **catalog-first** — you don't write to a path, you write to a *table* in a catalog. The catalog tracks which files belong to the current snapshot.

We'll use the simplest catalog: a **Hadoop catalog** rooted at a local folder. No Hive metastore, no extra services.

**Compatibility note**: Spark **4.1.4 / Scala 2.13** → `iceberg-spark-runtime-4.1_2.13:1.12.0`. Match your Spark major.minor version in the artifact name.

First run downloads the jar via `spark.jars.packages` — be patient on the first cell.

## Wipe stale warehouse first

If a previous run left a partial `<table>/` folder under the warehouse, the Hadoop catalog fails with:

```
FileNotFoundException: File .../metadata/version-hint.text does not exist
```

`version-hint.text` is the Hadoop catalog's pointer to the current metadata file — if the directory exists but the metadata is missing, the catalog won't recover on its own. Wiping the warehouse is the cleanest reset for a tutorial notebook. Skip this cell if you want to keep prior tables.

In [ ]:
! rm -rf /tmp/iceberg_warehouse

In [ ]:
from pyspark.sql import SparkSession, functions as F

In [ ]:
warehouse = "file:///tmp/iceberg_warehouse"

spark = (
    SparkSession.builder
    .appName("Write Iceberg")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Define a catalog called 'local'
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", warehouse)
    .getOrCreate()
)

In [ ]:
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()
df.limit(5).toPandas()

## 1. Create namespace (database) and write the table

Three-part name: `<catalog>.<namespace>.<table>` → `local.db.mall_customers`.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

In [ ]:
# DataFrameWriterV2 is the recommended API for Iceberg
df.writeTo("local.db.mall_customers").using("iceberg").createOrReplace()

In [ ]:
# Iceberg lays it out as warehouse/<namespace>/<table>/{data,metadata}
! ls -lh /tmp/iceberg_warehouse/db/mall_customers
! ls -lh /tmp/iceberg_warehouse/db/mall_customers/metadata

## 2. Read it back

In [ ]:
spark.table("local.db.mall_customers").count()

In [ ]:
spark.sql("SELECT * FROM local.db.mall_customers LIMIT 5").show()

## 3. Partitioned table

In [ ]:
(
    df.writeTo("local.db.mall_customers_by_gender")
      .using("iceberg")
      .partitionedBy(F.col("Gender"))
      .createOrReplace()
)

In [ ]:
! ls /tmp/iceberg_warehouse/db/mall_customers_by_gender/data

## 4. Append

In [ ]:
df_more = df.limit(5)
df_more.writeTo("local.db.mall_customers").append()

spark.table("local.db.mall_customers").count()

## 5. Snapshots and time travel

Each commit creates a new snapshot. Iceberg exposes them as a metadata table `<table>.snapshots`.

In [ ]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM local.db.mall_customers.snapshots").show(truncate=False)

In [ ]:
# Read a previous snapshot — grab one from the cell above
first_snapshot = spark.sql(
    "SELECT snapshot_id FROM local.db.mall_customers.snapshots ORDER BY committed_at LIMIT 1"
).collect()[0][0]

spark.read \
    .option("versionAsOf", first_snapshot) \
    .table("local.db.mall_customers") \
    .count()

## 6. Update / Delete via SQL

In [ ]:
spark.sql("""
    UPDATE local.db.mall_customers
    SET SpendingScore = SpendingScore + 1
    WHERE Gender = 'Female'
""")

In [ ]:
spark.sql("""
    DELETE FROM local.db.mall_customers
    WHERE Age > 65
""")

## 7. Inspect history and files

In [ ]:
spark.sql("SELECT * FROM local.db.mall_customers.history").show(truncate=False)

In [ ]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM local.db.mall_customers.files").show(truncate=False)

In [ ]:
spark.stop()